In [1]:
import pandas as pd
import numpy as np
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans

# ==========================================
# 1. CREATE SYNTHETIC AAA WRESTLING DATASET
# ==========================================
# Defining key aspects of Lucha Libre AAA to populate our text descriptions
luchadores = ["Psycho Clown", "Pentagon Jr.", "Rey Fenix", "Pagano", "El Hijo del Vikingo", "Chessman", "Cibernetico", "La Parka"]
styles = ["Retro Vintage", "Gothic Dark", "Neon Streetwear", "Traditional Mask", "High-Flying Graphic", "Hardcore Extreme"]
colors = ["Black", "White", "Red", "Neon Green", "Purple", "Yellow"]

np.random.seed(42)
data = []

# Generate 200 sample merchandise entries
for i in range(1, 201):
    wrestler = np.random.choice(luchadores)
    style = np.random.choice(styles)
    color = np.random.choice(colors)
    price = round(np.random.uniform(19.99, 45.00), 2)
    
    # Crafting varied textual merchandise descriptions mimicking an online storefront
    templates = [
        f"Official AAA {wrestler} wrestling T-shirt featuring a bold {style} graphic. Premium {color} cotton tee.",
        f"Lucha Libre AAA Worldwide merchandise celebrating {wrestler}. {style} theme printed on a comfortable {color} shirt.",
        f"Authentic {style} edition {wrestler} TV show tee. Vibrant artwork on durable {color} fabric.",
        f"Celebrate AAA TV with this {wrestler} graphic tee. Featuring an iconic {style} layout in {color}."
    ]
    description = np.random.choice(templates)
    
    data.append({
        "product_id": f"AAA-TS-{i:03d}",
        "wrestler": wrestler,
        "style_category": style,
        "base_color": color,
        "price_usd": price,
        "description": description
    })

df = pd.DataFrame(data)
print("--- Initial Dataset Sample ---")
print(df[["product_id", "description"]].head(3))
print("\n" + "="*50 + "\n")


# ==========================================
# 2. NLP PREPROCESSING FUNCTION
# ==========================================
# Hand-rolled basic cleaning to avoid bulky dependencies like NLTK or SpaCy
STOPWORDS = {"the", "a", "an", "and", "or", "but", "with", "on", "in", "at", "for", "this", "of", "featuring"}

def preprocess_text(text):
    # Lowercase
    text = text.lower()
    # Remove special characters/punctuation
    text = re.sub(r'[^a-z0-9\s-]', '', text)
    # Tokenize and remove stop words
    words = text.split()
    cleaned_words = [word for word in words if word not in STOPWORDS]
    return " ".join(cleaned_words)

df["cleaned_description"] = df["description"].apply(preprocess_text)


# ==========================================
# 3. NLP FEATURE EXTRACTION (TF-IDF)
# ==========================================
# Converting text columns into numerical vectors
vectorizer = TfidfVectorizer(max_features=100)
tfidf_matrix = vectorizer.fit_transform(df["cleaned_description"])


# ==========================================
# 4. K-MEANS CLUSTERING
# ==========================================
# We cluster descriptions into 4 semantic merchandise groupings
num_clusters = 4
kmeans = KMeans(n_clusters=num_clusters, init='k-means++', max_iter=300, random_state=42)
df["cluster"] = kmeans.fit_predict(tfidf_matrix)


# ==========================================
# 5. PROFILE AND EVALUATE CLUSTERS
# ==========================================
print("--- Cluster Profiling ---")
order_centroids = kmeans.cluster_centers_.argsort()[:, ::-1]
terms = vectorizer.get_feature_names_out()

for cluster_id in range(num_clusters):
    print(f"\nCluster {cluster_id}:")
    
    # Get top 5 terms driving this specific cluster
    top_terms = [terms[ind] for ind in order_centroids[cluster_id, :5]]
    print(f"  > Top keywords: {', '.join(top_terms)}")
    
    # Display top representative wrestlers and metadata mapping back to this cluster
    cluster_sample = df[df["cluster"] == cluster_id]
    top_wrestlers = cluster_sample["wrestler"].value_counts().head(2).index.tolist()
    top_styles = cluster_sample["style_category"].value_counts().head(2).index.tolist()
    
    print(f"  > Common Luchadores: {', '.join(top_wrestlers)}")
    print(f"  > Dominated Styles: {', '.join(top_styles)}")
    print(f"  > Sample Item: '{cluster_sample['description'].iloc[0]}'")


--- Initial Dataset Sample ---
   product_id                                        description
0  AAA-TS-001  Celebrate AAA TV with this Cibernetico graphic...
1  AAA-TS-002  Authentic High-Flying Graphic edition El Hijo ...
2  AAA-TS-003  Authentic High-Flying Graphic edition Rey Feni...


--- Cluster Profiling ---

Cluster 0:
  > Top keywords: bold, cotton, wrestling, official, premium
  > Common Luchadores: Psycho Clown, El Hijo del Vikingo
  > Dominated Styles: Traditional Mask, Retro Vintage
  > Sample Item: 'Official AAA Pentagon Jr. wrestling T-shirt featuring a bold Hardcore Extreme graphic. Premium Purple cotton tee.'

Cluster 1:
  > Top keywords: layout, celebrate, iconic, graphic, tv
  > Common Luchadores: Psycho Clown, Chessman
  > Dominated Styles: High-Flying Graphic, Traditional Mask
  > Sample Item: 'Celebrate AAA TV with this Cibernetico graphic tee. Featuring an iconic Traditional Mask layout in Purple.'

Cluster 2:
  > Top keywords: artwork, durable, edition, vibran